# IS630: Green Light for an Indie Studio
## Notebook 02: Cleaning and Feature Engineering
**Group**: [Insert names]

> **AI Coding Assistant Declaration**:
> In accordance with course guidelines, an AI coding assistant was used strictly for code scaffolding and plotting functions. All code blocks below are labeled as AI-assisted. All modelling decisions, interpretations, and written commentaries are independently authored by group members in the designated sections below.

This notebook works through one data question per section, building towards **AQ2**


- Apply exclusions decided in 01: software, zero-review games, implausible playtime, free-to-play (decide)
- Story-driven flag, with the rule written down
- Build the target(s) chosen for AQ1
- Encode features: genres, price bands, skewed-variable transforms, game age
- Fix inclusion thresholds before any model is fitted
- Split a validation set out of train
- Save train, validation and the feature spec

*Outline written by AI Assistant*

In [1]:
# import libraries and load data
import sys
import re
import ast
from pathlib import Path

import numpy as np 
import pandas as pd
from IPython.display import display

# Locate the project root
ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "src" / "is630").exists()
)

sys.path.insert(0, str(ROOT / "src"))

from is630 import config

# Load the output from notebook 01
input_path = config.INTERIM / "01_eda.parquet"

if not input_path.exists():
    raise FileNotFoundError(
        "Run Notebook 01 and save 01_eda.parquet first"
    )

df = pd.read_parquet(input_path)

required_columns = [
    "steam_appid",
    "steam_is_software",
    "steam_positive",
    "steam_negative",
    "steam_review_count",
    "steam_price",
    "steam_playtime_hours",
    "matched_hltb",
    "hltb_main_story",
]

missing = set(required_columns) - set(df.columns)
assert not missing, f"Missing required columns: {missing}"

print(f"Dataset loaded: {len(df):,} rows")
print(f"Number of columns: {df.shape[1]}")
print(f"Unique games: {df['steam_appid'].nunique():,}")

display(df.head())

Dataset loaded: 97,026 rows
Number of columns: 83
Unique games: 97,026


,steam_appid,steam_name,steam_release_date,steam_estimated_owners,steam_peak_ccu,steam_required_age,steam_price,steam_discount,steam_dlc_count,steam_about_the_game,...,steam_positive_rate,steam_playtime_hours,steam_years_since_release,steam_language_count,steam_has_metacritic,completion_ratio,steam_owners_lower,steam_owners_upper,steam_is_software,steam_review_bucket
0,730,Counter-Strike 2,"Aug 21, 2012",100000000 - 200000000,1013936,0,0.00,0,1,"For over two decades, Counter-Strike has offer...",...,0.866932,103.950000,14,29,False,NaN,100000000,200000000,False,20k+
1,570,Dota 2,"Jul 9, 2013",100000000 - 200000000,623941,0,0.00,0,2,"The most-played game on Steam. Every day, mill...",...,0.815193,19.250000,13,28,True,NaN,100000000,200000000,False,20k+
2,271590,Grand Theft Auto V Legacy,"Apr 13, 2015",50000000 - 100000000,67851,17,0.00,0,0,"When a young street hustler, a retired bank ro...",...,0.874118,97.433333,11,13,True,NaN,50000000,100000000,False,20k+
3,105600,Terraria,"May 16, 2011",20000000 - 50000000,24580,0,4.99,50,2,"Dig, Fight, Explore, Build: The very world is ...",...,0.974818,34.183333,15,9,True,0.660675,20000000,50000000,False,20k+
4,359550,Tom Clancy's Rainbow Six® Siege X,"Dec 1, 2015",20000000 - 50000000,51847,17,0.00,0,9,Tom Clancy's Rainbow Six Siege X is the undisp...,...,0.838601,41.700000,11,16,False,NaN,20000000,50000000,False,20k+


### 1.1 Initial Data Quality Assessment

An initial data quality assessment was conducted to identify observations requiring exclusion or special treatment before statistical modelling.

Of the 97,026 observations in the training dataset, 1,329 (1.37%) were classified as non-game software and were identified for exclusion.

A total of 32,477 observations (33.47%) had zero Steam reviews, making their positive review rates undefined. These observations will be retained in the cleaned base dataset but excluded from analyses requiring an observed review proportion.

Furthermore, 76,364 observations (78.70%) had zero recorded Steam playtime. Since these observations cannot provide a meaningful positive completion ratio, they will be excluded from completion-ratio modelling while remaining available for other analyses.

Only 22 games had median playtime exceeding 1,000 hours, including one exceeding 8,760 hours. Extreme observations will be flagged and evaluated using a predefined threshold.

Free-to-play games accounted for 20.73% of observations and will be retained to preserve their representation in the Steam market.

No negative or missing price values were identified.

These decisions establish a consistent approach to data cleaning while minimizing unnecessary information loss and preserving observations for different analytical objectives.

In [2]:
# Inspect candidate exclusions before removing anything

review_count = pd.to_numeric(
    df["steam_review_count"], errors="coerce"
)

playtime = pd.to_numeric(
    df["steam_playtime_hours"], errors="coerce"
)

price = pd.to_numeric(
    df["steam_price"], errors="coerce"
)

audit = pd.DataFrame({
    "Condition": [
        "Non-game software",
        "Zero-review games",
        "Zero Steam playtime",
        "Median playtime > 1,000 hours",
        "Median playtime > 8,760 hours",
        "Free-to-play games",
        "Negative prices",
        "Missing prices"
    ],
    "Number of games": [
        df["steam_is_software"].fillna(False).sum(),
        review_count.eq(0).sum(),
        playtime.eq(0).sum(),
        playtime.gt(1000).sum(),
        playtime.gt(8760).sum(),
        price.eq(0).sum(),
        price.lt(0).sum(),
        price.isna().sum()
    ]
})

audit["Percentage (%)"] = (
    audit["Number of games"] / len(df) * 100
).round(2)

display(audit)


,Condition,Number of games,Percentage (%)
0,Non-game software,1329,1.37
1,Zero-review games,32477,33.47
2,Zero Steam playtime,76364,78.70
3,"Median playtime > 1,000 hours",22,0.02
4,"Median playtime > 8,760 hours",1,0.00
5,Free-to-play games,20117,20.73
6,Negative prices,0,0.00
7,Missing prices,0,0.00


In [3]:

# Identify invalid observations

positive = pd.to_numeric(
    df["steam_positive"], errors="coerce"
)

negative = pd.to_numeric(
    df["steam_negative"], errors="coerce"
)

software = df["steam_is_software"].fillna(False)

invalid_reviews = (
    positive.isna()
    | negative.isna()
    | positive.lt(0)
    | negative.lt(0)
)

invalid_price = price.lt(0)

# Exclude non-game software and invalid records
exclude = software | invalid_reviews | invalid_price

df_clean = df.loc[~exclude].copy()

# Define eligibility flags for subsequent stages
df_clean["flag_zero_reviews"] = (
    df_clean["steam_review_count"] == 0
)

df_clean["flag_free_to_play"] = (
    df_clean["steam_price"] == 0
)

hours = pd.to_numeric(
    df_clean["steam_playtime_hours"], errors="coerce"
)

df_clean["flag_extreme_playtime"] = (
    hours.lt(0) | hours.gt(8760)
)

# Keep only positive, non-extreme playtime for
# future completion-ratio calculations
df_clean["steam_playtime_hours_usable"] = hours.where(
    hours.gt(0) & hours.le(8760)
)

print(f"Original games: {len(df):,}")
print(f"Excluded games: {exclude.sum():,}")
print(f"Retained games: {len(df_clean):,}")

print(
    "Retained games with reviews:",
    f"{(~df_clean['flag_zero_reviews']).sum():,}"
)

print(
    "Retained free-to-play games:",
    f"{df_clean['flag_free_to_play'].sum():,}"
)

print(
    "Usable Steam playtime observations:",
    f"{df_clean['steam_playtime_hours_usable'].notna().sum():,}"
)

assert df_clean["steam_appid"].is_unique


Original games: 97,026
Excluded games: 1,329
Retained games: 95,697
Retained games with reviews: 63,554
Retained free-to-play games: 19,677
Usable Steam playtime observations: 20,395


In [4]:

cleaning_summary = pd.DataFrame({
    "Stage": [
        "Initial dataset",
        "Non-game software removed",
        "Invalid records removed (including overlaps)",
        "Cleaned base dataset"
    ],
    "Games": [
        len(df),
        int(software.sum()),
        int((exclude & ~software).sum()),
        len(df_clean)
    ]
})

cleaning_summary["Percentage of original"] = (
    cleaning_summary["Games"] / len(df) * 100
).round(2)

display(cleaning_summary)

assert len(df_clean) + int(exclude.sum()) == len(df)


,Stage,Games,Percentage of original
0,Initial dataset,97026,100.00
1,Non-game software removed,1329,1.37
2,Invalid records removed (including overlaps),0,0.00
3,Cleaned base dataset,95697,98.63


In [5]:
# Step 2: Inspect metadata used for story-driven classification

metadata_cols = [
    "steam_name",
    "steam_genres",
    "steam_categories",
    "steam_tags"
]

display(df_clean[metadata_cols].sample(
    10, random_state=config.SEED
))

print("Missing metadata values:")
print(
    df_clean[metadata_cols]
    .isna()
    .sum()
)

print("\nSingle-player games:")
print(
    df_clean["steam_categories"]
    .fillna("")
    .str.contains("Single-player", case=False, regex=False)
    .sum()
)


,steam_name,steam_genres,steam_categories,steam_tags
90125,Testimony - Metal Graveyard,Adventure,"Single-player,Family Sharing",None
96809,Heavy Hearts,"Action,Indie,RPG","Single-player,Partial Controller Support,Steam...",None
21508,Astral Masters,"Indie,Strategy","Single-player,Multi-player,PvP,Online PvP,LAN ...","Strategy,Card Game,Turn-Based Strategy,Fantasy..."
66119,101 Cats Hidden in Bangkok,"Casual,Indie","Single-player,Steam Achievements,Steam Cloud,F...","Cats,Hidden Object,Wholesome,Puzzle,Cozy,Casua..."
61706,Elya Richi vs Blackstone : Nuclear War,Indie,"Single-player,Multi-player,PvP,Shared/Split Sc...","Simulation,Strategy,Action,Point & Click,Arcad..."
91841,Ananas - Pineapple Idle Game,"Casual,Indie,Simulation,Free To Play","Single-player,Steam Achievements,Steam Cloud",None
66000,Divinity Chess,Strategy,"Single-player,Family Sharing",None
33280,The Slothgate Experiment,"Action,Indie","Single-player,Tracked Controller Support,VR On...","Action,Puzzle,Immersive Sim,3D,Realistic,VR,At..."
18226,Bloody Knuckles Street Boxing,"Action,Indie,Simulation,Sports,Strategy,Early ...","Single-player,Multi-player,PvP,Shared/Split Sc...","Early Access,Boxing,Simulation,Arcade,Sports,C..."
5085,SONG OF HORROR COMPLETE EDITION,Adventure,"Single-player,Steam Achievements,Full controll...","Survival Horror,Dark,Horror,Psychological Horr..."


Missing metadata values:
steam_name              0
steam_genres         6320
steam_categories     6049
steam_tags          31837
dtype: int64

Single-player games:
85700


### 1.2 Story-Driven Game Classification

A rule-based classification approach was implemented to identify story-driven games using Steam categories, tags, and genres. Games were classified as story-driven if they supported single-player gameplay and contained narrative-related indicators such as `Story Rich`, `Visual Novel`, `Interactive Fiction`, or `Choices Matter`.

Of the 95,697 cleaned observations, 85,700 (89.55%) supported single-player gameplay, while 15,012 (15.69%) contained narrative-related tags. Applying the classification criteria identified **14,863 story-driven games (15.53%)**.

Among these games, 5,118 (34.43%) were successfully matched with HowLongToBeat (HLTB) records and had positive main-story completion times, providing a potential sample for completion-related analysis.

A metadata quality assessment identified 31,873 games with missing category or tag information. None of the classified story-driven games had incomplete metadata. However, missing information may cause some story-driven games to remain unidentified.

Finally, logical consistency checks confirmed that all classified story-driven games satisfied the single-player requirement and that the classification count matched the expected total. All validation checks passed.

In [35]:

import re

# Narrative-related Steam tags
STORY_TAGS = [
    "Story Rich",
    "Visual Novel",
    "Interactive Fiction",
    "Narrative",
    "Choices Matter",
    "Choose Your Own Adventure"
]

# Normalize relevant metadata to strings
categories = df_clean["steam_categories"].fillna("").astype(str)
tags = df_clean["steam_tags"].fillna("").astype(str)
genres = df_clean["steam_genres"].fillna("").astype(str)

# Identify single-player games
df_clean["flag_single_player"] = categories.str.contains(
    "Single-player",
    case=False,
    regex=False
)

# Match complete narrative tag names, not arbitrary substrings
tag_pattern = (
    r"(?<![A-Za-z0-9])(?:"
    + "|".join(re.escape(tag) for tag in STORY_TAGS)
    + r")(?![A-Za-z0-9])"
)

df_clean["flag_narrative_tags"] = tags.str.contains(
    tag_pattern,
    case=False,
    regex=True
)

# Visual Novel can also appear in genre metadata
df_clean["flag_visual_novel_genre"] = genres.str.contains(
    r"(?<![A-Za-z0-9])Visual Novel(?![A-Za-z0-9])",
    case=False,
    regex=True
)


from is630.features import classify_story_driven

# Final story-driven classification using the shared function
df_clean["flag_story_driven"] = classify_story_driven(df_clean)


print("Story-driven classification completed.")
print(
    f"Story-driven games: "
    f"{df_clean['flag_story_driven'].sum():,}"
)

print(
    f"Percentage of cleaned dataset: "
    f"{df_clean['flag_story_driven'].mean() * 100:.2f}%"
)


Story-driven classification completed.
Story-driven games: 14,863
Percentage of cleaned dataset: 15.53%


In [39]:

import sys
import importlib
from pathlib import Path

# Locate the shared features module
current = Path.cwd().resolve()

project_root = next(
    (
        folder for folder in [current, *current.parents]
        if (folder / "src" / "is630" / "features.py").exists()
    ),
    None
)

assert project_root is not None, "Project root not found"

sys.path.insert(0, str(project_root / "src"))

import is630.features as features
importlib.reload(features)

# Compare original and refactored classifications
original = df_clean["flag_story_driven"]
refactored = features.classify_story_driven(df_clean)

print("Original story-driven count:", original.sum())
print("Refactored story-driven count:", refactored.sum())
print("Mismatched classifications:", (original != refactored).sum())

assert original.equals(refactored)

print("All classifications match!")


Original story-driven count: 14863
Refactored story-driven count: 14863
Mismatched classifications: 0
All classifications match!


In [40]:

story_summary = pd.DataFrame({
    "Condition": [
        "All cleaned games",
        "Single-player games",
        "Games with narrative tags",
        "Games with Visual Novel genre",
        "Flagged story-driven games",
        "Story-driven and HLTB matched",
        "Story-driven with positive HLTB main-story time"
    ],
    "Number of games": [
        len(df_clean),
        int(df_clean["flag_single_player"].sum()),
        int(df_clean["flag_narrative_tags"].sum()),
        int(df_clean["flag_visual_novel_genre"].sum()),
        int(df_clean["flag_story_driven"].sum()),
        int((
            df_clean["flag_story_driven"]
            & df_clean["matched_hltb"].fillna(False)
        ).sum()),
        int((
            df_clean["flag_story_driven"]
            & (df_clean["hltb_main_story"] > 0)
        ).sum())
    ]
})

story_summary["Percentage of cleaned (%)"] = (
    story_summary["Number of games"] / len(df_clean) * 100
).round(2)

display(story_summary)

# Inspect example classifications
display(
    df_clean.loc[
        df_clean["flag_story_driven"],
        [
            "steam_name",
            "steam_genres",
            "steam_tags",
            "steam_review_count",
            "matched_hltb"
        ]
    ].sample(
        min(10, int(df_clean["flag_story_driven"].sum())),
        random_state=config.SEED
    )
)


,Condition,Number of games,Percentage of cleaned (%)
0,All cleaned games,95697,100.00
1,Single-player games,85700,89.55
2,Games with narrative tags,15012,15.69
3,Games with Visual Novel genre,0,0.00
4,Flagged story-driven games,14863,15.53
5,Story-driven and HLTB matched,5118,5.35
6,Story-driven with positive HLTB main-story time,5118,5.35


,steam_name,steam_genres,steam_tags,steam_review_count,matched_hltb
27668,INFINITE GUITARS,"Action,Adventure,Indie,RPG","RPG,Adventure,Action,Indie,Rhythm,Turn-Based,T...",35,True
7981,Yasha: Legends of the Demon Blade,"Action,Adventure,Indie,RPG","Action Roguelike,Exploration,Rogue-lite,Single...",530,True
36455,Escape from the Cannibal Family,"Adventure,Indie","Adventure,Horror,Stealth,Psychological Horror,...",17,False
13014,Go For A Punch! Saki Sanobashi,"Action,Adventure,Casual,Indie","Choose Your Own Adventure,Clicker,Interactive ...",203,False
55494,Five Days of Hell,"Adventure,Indie","Adventure,Rogue-like,Action-Adventure,Action R...",3,False
7931,STAY,"Adventure,Indie","Indie,Adventure,Choices Matter,Story Rich,Puzz...",536,True
11473,Inbetween Land,"Adventure,Casual,Indie","Casual,Adventure,Indie,Hidden Object,Point & C...",266,True
8473,Chernobylite 2: Exclusion Zone,"Action,Adventure,Indie,RPG,Early Access","Early Access,Adventure,Choices Matter,Open Wor...",473,False
22933,Untale: King of Revinia,"Adventure,Indie","Story Rich,Mystery,Visual Novel,Drama,Choices ...",55,False
26618,Survive Till Morning,Indie,"Indie,Dark Fantasy,Puzzle,Atmospheric,Horror,S...",39,False


In [41]:

# Record missing metadata that may affect classification

df_clean["flag_missing_story_metadata"] = (
    df_clean["steam_categories"].isna()
    | df_clean["steam_tags"].isna()
)

print(
    "Games with incomplete story metadata:",
    f"{df_clean['flag_missing_story_metadata'].sum():,}"
)

print(
    "Story-driven games with incomplete metadata:",
    f"{(df_clean['flag_story_driven'] & df_clean['flag_missing_story_metadata']).sum():,}"
)

# Verify logical consistency
assert not (
    df_clean["flag_story_driven"]
    & ~df_clean["flag_single_player"]
).any()

assert (
    df_clean["flag_story_driven"].sum() > 0
)

print("Story-driven classification checks passed.")


Games with incomplete story metadata: 31,873
Story-driven games with incomplete metadata: 0
Story-driven classification checks passed.


In [42]:

# Record missing metadata that may affect classification

df_clean["flag_missing_story_metadata"] = (
    df_clean["steam_categories"].isna()
    | df_clean["steam_tags"].isna()
)

print(
    "Games with incomplete story metadata:",
    f"{df_clean['flag_missing_story_metadata'].sum():,}"
)

print(
    "Story-driven games with incomplete metadata:",
    f"{(df_clean['flag_story_driven'] & df_clean['flag_missing_story_metadata']).sum():,}"
)

# Verify logical consistency
assert not (
    df_clean["flag_story_driven"]
    & ~df_clean["flag_single_player"]
).any()


assert (
    df_clean["flag_story_driven"].sum() > 0
), "No story-driven games identified."


print("Story-driven classification checks passed.")


Games with incomplete story metadata: 31,873
Story-driven games with incomplete metadata: 0
Story-driven classification checks passed.


### 1.3 Target Variable Construction

Three target variables were constructed to support subsequent statistical modelling: Steam review count, positive review rate, and completion ratio.

Review count was calculated as the sum of positive and negative reviews and was available for all 95,697 games. The median review count was 6, compared to a mean of 1,208.29, indicating a strongly right-skewed distribution.

Positive review rate was calculated as the proportion of positive reviews among total reviews. This was available for 63,554 games (66.41%), with a median of 0.820. Games with zero reviews were assigned missing values because their review proportions were undefined.

Completion ratio was calculated by dividing median Steam playtime by HLTB main-story duration for eligible story-driven games. A total of 3,508 games (3.67%) satisfied these requirements, with a median completion ratio of 0.831 and a mean of 2.270.

Eligibility flags were created for subsequent modelling, and validation checks confirmed that all target variables satisfied their expected value constraints.

The final modelling thresholds will be established before model fitting to ensure consistent sample selection.

In [43]:
# Step 3: Construct target variables

# Target 1: Review count
positive = pd.to_numeric(
    df_clean["steam_positive"], errors="coerce"
)

negative = pd.to_numeric(
    df_clean["steam_negative"], errors="coerce"
)


# Target 2: Completion ratio
# HLTB main-story time is measured in hours
main_story = pd.to_numeric(
    df_clean["hltb_main_story"], errors="coerce"
)

# Identify games with sufficient data to calculate the ratio
completion_eligible = (
    df_clean["flag_story_driven"]
    & df_clean["matched_hltb"].fillna(False)
    & main_story.gt(0)
    & df_clean["steam_playtime_hours_usable"].gt(0)
)


# Eligibility flags for subsequent modelling
df_clean["flag_reception_eligible"] = (
    df_clean["steam_review_count"] > 0
)


df_clean["flag_completion_eligible"] = (
    completion_eligible
    & df_clean["completion_ratio"].gt(0)
)

print("Target variables successfully constructed.")


Target variables successfully constructed.


In [44]:

# Summarize target availability

target_cols = [
    "steam_review_count",
    "steam_positive_rate",
    "completion_ratio"
]

target_summary = pd.DataFrame({
    "Target": target_cols,
    "Available games": [
        df_clean[col].notna().sum()
        for col in target_cols
    ],
    "Missing games": [
        df_clean[col].isna().sum()
        for col in target_cols
    ],
    "Median": [
        df_clean[col].median()
        for col in target_cols
    ],
    "Mean": [
        df_clean[col].mean()
        for col in target_cols
    ]
})

target_summary["Coverage (%)"] = (
    target_summary["Available games"]
    / len(df_clean) * 100
).round(2)

display(target_summary.round(3))

# Check target validity
assert df_clean["steam_review_count"].notna().all()
assert df_clean["steam_review_count"].ge(0).all()

assert df_clean["steam_positive_rate"].dropna().between(
    0, 1
).all()

assert (
    df_clean.loc[
        df_clean["flag_completion_eligible"],
        "completion_ratio"
    ].gt(0).all()
)



# Verify review-rate availability
assert (
    df_clean["steam_positive_rate"].notna()
    == df_clean["flag_reception_eligible"]
).all()

# Verify completion-ratio availability
assert (
    df_clean.loc[
        df_clean["flag_completion_eligible"],
        "completion_ratio"
    ].notna().all()
)

print("All target validation checks passed.")



,Target,Available games,Missing games,Median,Mean,Coverage (%)
0,steam_review_count,95697,0,6.000,1208.292,100.00
1,steam_positive_rate,63554,32143,0.820,0.760,66.41
2,completion_ratio,16653,79044,0.308,2.169,17.40


All target validation checks passed.


### 1.4 Feature Engineering and Encoding

Feature engineering was performed to prepare predictor variables for statistical modelling. Seven common Steam genres were converted into binary indicators, allowing games to belong to multiple genres. An additional indicator identified 6,320 games (6.60%) with missing genre information.

Steam prices were transformed using `log1p()` to reduce skewness and categorized into five price bands: Free, $0–5, $5–10, $10–20, and $20+. Game age was retained as a numerical predictor, ranging from 0 to 29 years.

The language-count feature was investigated after identifying unusually high counts of 103 languages. A revised parsing method processed all 95,697 records and identified 1,557 one-count discrepancies compared with the original calculation. These corrections were retained, although some uncertainty remains due to the use of fallback parsing.

Additionally, 6,289 games had zero recorded languages, which were treated as unavailable information rather than confirmed absence of language support. This left 89,408 observations with usable language counts, which were transformed using `log1p()`.

**Missing-value handling for language counts has not yet been completed.** These values were intentionally retained as `NaN` and will be imputed after splitting the data into training and validation sets in Step 6. The imputation value will be calculated using only the training data to prevent data leakage.

Finally, validation checks confirmed that genre indicators were binary, price-band assignments were complete, and the engineered numerical features satisfied their expected value constraints.

In [45]:
# Step 4: Inspect candidate predictor variables

numeric_features = [
    "steam_price",
    "steam_years_since_release",
    "steam_language_count"
]

# Check that the expected features are available
missing = set(numeric_features + ["steam_genres"]) - set(df_clean.columns)
assert not missing, f"Missing features: {missing}"

print("Numeric feature summary:")
display(
    df_clean[numeric_features].describe(
        percentiles=[0.25, 0.50, 0.75, 0.95, 0.99]
    ).T.round(2)
)

print("\nMissing values:")
display(df_clean[numeric_features + ["steam_genres"]].isna().sum())

# Examine the most common game genres
genre_counts = (
    df_clean["steam_genres"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
    .value_counts()
)

print("\nTop 15 genres:")
display(genre_counts.head(15))


Numeric feature summary:


,count,mean,std,min,25%,50%,75%,95%,99%,max
steam_price,95697.0,4.80,12.73,0.0,0.59,2.39,5.59,14.99,29.99,999.98
steam_years_since_release,95697.0,4.38,3.32,0.0,2.00,4.00,6.00,10.00,14.00,29.00
steam_language_count,95697.0,5.33,13.68,0.0,1.00,1.00,5.00,16.00,103.00,103.00



Missing values:


steam_price                     0
steam_years_since_release       0
steam_language_count            0
steam_genres                 6320
dtype: int64


Top 15 genres:


steam_genres
Indie                    63870
Casual                   39839
Action                   36746
Adventure                35455
Simulation               19214
Strategy                 17723
RPG                      16122
Free To Play              9287
Early Access              8790
Sports                    3906
Racing                    3265
Massively Multiplayer     1869
Violent                    313
Gore                       202
Education                  141
Name: count, dtype: int64

In [46]:
# Investigate unusual language counts

lang = df_clean["steam_language_count"]

print("Games with 103 languages:", lang.eq(103).sum())
print("Games with 0 languages:", lang.eq(0).sum())
print("Games with more than 30 languages:", lang.gt(30).sum())

print("\nMost common language counts:")
display(lang.value_counts().head(15))

# Examine the original source values
extreme_languages = df_clean.loc[
    lang.eq(103),
    [
        "steam_name",
        "steam_language_count",
        "steam_supported_languages"
    ]
]

print("\nExamples of games with 103 languages:")

for _, row in extreme_languages.head(5).iterrows():
    print("\nGame:", row["steam_name"])
    print("Language count:", row["steam_language_count"])
    print(
        "Original value:",
        repr(row["steam_supported_languages"])[:600]
    )


Games with 103 languages: 1530
Games with 0 languages: 6289
Games with more than 30 languages: 1763

Most common language counts:


steam_language_count
1      46111
2      11395
0       6289
3       4467
4       3246
5       2938
6       2394
9       2053
7       2048
8       2018
10      1930
11      1735
103     1530
12      1497
13      1126
Name: count, dtype: int64


Examples of games with 103 languages:

Game: Zup! S
Language count: 103
Original value: "['English', 'Russian', 'French', 'Italian', 'German', 'Spanish - Spain', 'Arabic', 'Bulgarian', 'Portuguese - Brazil', 'Hungarian', 'Greek', 'Danish', 'Traditional Chinese', 'Simplified Chinese', 'Korean', 'Dutch', 'Norwegian', 'Polish', 'Portuguese - Portugal', 'Romanian', 'Thai', 'Turkish', 'Ukrainian', 'Finnish', 'Czech', 'Swedish', 'Japanese', 'Spanish - Latin America', 'Vietnamese', 'Azerbaijani', 'Albanian', 'Amharic', 'Armenian', 'Assamese', 'Afrikaans', 'Basque', 'Belarusian', 'Bangla', 'Bosnian', 'Valencian', 'Welsh', 'Wolof', 'Galician', 'Georgian', 'Gujarati', 'Dari', 'Zulu', 'Hebr

Game: Zup! F
Language count: 103
Original value: "['English', 'Russian', 'French', 'Italian', 'German', 'Spanish - Spain', 'Arabic', 'Bulgarian', 'Portuguese - Brazil', 'Hungarian', 'Vietnamese', 'Greek', 'Danish', 'Spanish - Latin America', 'Traditional Chinese', 'Simplified Chinese', 'Korean', 'Dutch', 'No

In [47]:

# Step 4: Feature engineering and encoding

# A. Genre encoding
# Select common genres identified during EDA


GENRE_MAP = {
    "Indie": "genre_indie",
    "Casual": "genre_casual",
    "Action": "genre_action",
    "Adventure": "genre_adventure",
    "Simulation": "genre_simulation",
    "Strategy": "genre_strategy",
    "RPG": "genre_rpg",
    "Early Access": "genre_early_access"
}


# Convert comma-separated genres to sets
genre_sets = df_clean["steam_genres"].fillna("").apply(
    lambda x: {g.strip() for g in x.split(",")}
)

for genre, column in GENRE_MAP.items():
    df_clean[column] = genre_sets.apply(
        lambda genres: int(genre in genres)
    )

# Indicator for missing genre information
df_clean["flag_missing_genre"] = (
    df_clean["steam_genres"].isna().astype(int)
)

# B. Price transformations
df_clean["feature_log_price"] = np.log1p(
    df_clean["steam_price"]
)

# Use fixed price bands in USD
PRICE_BINS = [-0.01, 0, 5, 10, 20, np.inf]
PRICE_LABELS = [
    "Free",
    "0-5",
    "5-10",
    "10-20",
    "20+"
]

df_clean["feature_price_band"] = pd.cut(
    df_clean["steam_price"],
    bins=PRICE_BINS,
    labels=PRICE_LABELS,
    include_lowest=True
)

# One-hot encode price bands
price_dummies = pd.get_dummies(
    df_clean["feature_price_band"],
    prefix="price_band",
    dtype=int
)

# Remove existing price-band dummy columns before recreating them
df_clean = df_clean.loc[
    :, ~df_clean.columns.str.startswith("price_band_")
].copy()

df_clean = pd.concat(
    [df_clean, price_dummies],
    axis=1
)


# C. Q4 release indicator (October–December)
release_dates = pd.to_datetime(
    df_clean["steam_release_date"],
    errors="coerce",
    format="mixed"
)

df_clean["flag_q4_release"] = (
    release_dates.dt.month.isin([10, 11, 12])
    .astype(int)
)

# D. OS / Platform Support
df_clean["steam_mac"] = df_clean["steam_mac"].astype(int)
df_clean["steam_linux"] = df_clean["steam_linux"].astype(int)


# E. Audio Localization
from is630.features import list_length

# Count available full-audio languages
df_clean["steam_audio_language_count"] = list_length(
    df_clean["steam_full_audio_languages"]
)

# Log-transform audio language count
df_clean["feature_log_audio_language_count"] = np.log1p(
    df_clean["steam_audio_language_count"]
)

# Flag games offering full-audio localization
df_clean["flag_has_full_audio"] = (
    df_clean["steam_audio_language_count"] > 0
).astype(int)


# F. Achievements
assert df_clean["steam_achievements"].ge(0).all()

df_clean["feature_log_achievements"] = np.log1p(
    df_clean["steam_achievements"]
)

df_clean["flag_has_achievements"] = (
    df_clean["steam_achievements"] > 0
).astype(int)


# G. Age Restriction
df_clean["flag_age_restricted"] = (
    pd.to_numeric(
        df_clean["steam_required_age"],
        errors="coerce"
    ).fillna(0).gt(0)
).astype(int)


# H. Steam Category Indicators

# Split category strings into individual category names
category_sets = df_clean["steam_categories"].fillna("").apply(
    lambda value: {
        item.strip().casefold()
        for item in str(value).split(",")
        if item.strip()
    }
)

# Define the seven required category indicators
CATEGORY_FLAGS = {
    "cat_singleplayer": ["Single-player"],
    "cat_multiplayer": ["Multi-player"],
    "cat_coop": [
        "Co-op",
        "Online Co-op",
        "LAN Co-op",
        "Shared/Split Screen Co-op"
    ],
    "cat_controller": [
        "Full controller support",
        "Partial Controller Support",
        "Controller Support"
    ],
    "cat_steam_cloud": ["Steam Cloud"],
    "cat_trading_cards": ["Steam Trading Cards"],
    "cat_vr": ["VR Supported", "VR Only", "VR Support"]
}

# Create binary indicators
for flag, categories in CATEGORY_FLAGS.items():
    category_names = {name.casefold() for name in categories}

    df_clean[flag] = category_sets.apply(
        lambda values: int(bool(values & category_names))
    )


# I. Missing Steam Tags Indicator
df_clean["flag_missing_tags"] = (
    df_clean["steam_tags"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["", "[]", "none", "nan"])
    .astype(int)
)


# J. Self-Publishing Indicator

developers = (
    df_clean["steam_developers"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.casefold()
)

publishers = (
    df_clean["steam_publishers"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.casefold()
)

df_clean["flag_self_published"] = (
    developers.ne("")
    & publishers.ne("")
    & developers.eq(publishers)
).astype(int)


# K. Storefront / Support Presence

# Identify non-empty metadata values
def has_value(series):
    values = (
        series.fillna("")
        .astype(str)
        .str.strip()
        .str.casefold()
    )
    return ~values.isin(["", "none", "nan"])

# Website presence
df_clean["flag_has_website"] = (
    has_value(df_clean["steam_website"])
).astype(int)

# Support URL or email presence
df_clean["flag_has_support_contact"] = (
    has_value(df_clean["steam_support_url"])
    | has_value(df_clean["steam_support_email"])
).astype(int)


# L. HLTB Game Scope Features

# Log-transform main-story completion hours
df_clean["feature_log_main_story_hours"] = np.log1p(
    df_clean["hltb_main_story"]
)

# Identify games with additional side content
df_clean["flag_has_side_content"] = (
    df_clean["hltb_main_plus_sides"]
    > df_clean["hltb_main_story"]
).astype(int)



print("Feature engineering completed.")


Feature engineering completed.


In [48]:

print("Unique column names:", df_clean.columns.is_unique)

print(
    "Price-band dummy columns:",
    df_clean.columns.str.startswith("price_band_").sum()
)


Unique column names: True
Price-band dummy columns: 5


In [49]:

# Verify feature engineering results

feature_cols = (
    list(GENRE_MAP.values())
    + ["flag_missing_genre"]
    + ["feature_log_price", "steam_years_since_release"]
)

print("Engineered feature summary:")
display(df_clean[feature_cols].describe().T.round(3))

print("\nPrice band distribution:")
display(
    df_clean["feature_price_band"]
    .value_counts()
    .sort_index()
)

print("\nMissing values in engineered features:")
display(df_clean[feature_cols].isna().sum())

# Validation checks
assert df_clean["feature_log_price"].ge(0).all()
assert df_clean["steam_years_since_release"].ge(0).all()


assert df_clean["feature_price_band"].notna().all()

assert (
    df_clean[list(GENRE_MAP.values())]
    .isin([0, 1])
    .all()
    .all()
)

assert (
    price_dummies.sum(axis=1) == 1
).all()

print("\nAll feature encoding checks passed.")


Engineered feature summary:


,count,mean,std,min,25%,50%,75%,max
genre_indie,95697.0,0.667,0.471,0.0,0.000,1.000,1.000,1.000
genre_casual,95697.0,0.416,0.493,0.0,0.000,0.000,1.000,1.000
genre_action,95697.0,0.384,0.486,0.0,0.000,0.000,1.000,1.000
genre_adventure,95697.0,0.370,0.483,0.0,0.000,0.000,1.000,1.000
genre_simulation,95697.0,0.201,0.401,0.0,0.000,0.000,0.000,1.000
genre_strategy,95697.0,0.185,0.388,0.0,0.000,0.000,0.000,1.000
genre_rpg,95697.0,0.168,0.374,0.0,0.000,0.000,0.000,1.000
genre_early_access,95697.0,0.092,0.289,0.0,0.000,0.000,0.000,1.000
flag_missing_genre,95697.0,0.066,0.248,0.0,0.000,0.000,0.000,1.000
feature_log_price,95697.0,1.239,0.939,0.0,0.464,1.221,1.886,6.909



Price band distribution:


feature_price_band
Free     19677
0-5      51393
5-10     15355
10-20     7425
20+       1847
Name: count, dtype: int64


Missing values in engineered features:


genre_indie                  0
genre_casual                 0
genre_action                 0
genre_adventure              0
genre_simulation             0
genre_strategy               0
genre_rpg                    0
genre_early_access           0
flag_missing_genre           0
feature_log_price            0
steam_years_since_release    0
dtype: int64


All feature encoding checks passed.


In [50]:
# Parse the original language lists more reliably
def parse_language_count(value):
    if pd.isna(value) or not str(value).strip():
        return np.nan

    try:
        languages = ast.literal_eval(str(value))
    except (ValueError, SyntaxError):
        return np.nan

    if not isinstance(languages, (list, tuple)):
        return np.nan

    if not languages:
        return np.nan

    # Count unique language entries
    return len(set(str(x).strip() for x in languages))

df_clean["feature_language_count_parsed"] = (
    df_clean["steam_supported_languages"]
    .apply(parse_language_count)
)

# Compare original and parsed language counts
valid = df_clean["feature_language_count_parsed"].notna()

mismatches = (
    df_clean.loc[valid, "steam_language_count"]
    != df_clean.loc[valid, "feature_language_count_parsed"]
).sum()

print("Successfully parsed games:", valid.sum())
print("Missing or unparseable entries:", (~valid).sum())
print("Count mismatches:", mismatches)

print("\nGames originally recorded with 103 languages:")

display(
    df_clean.loc[
        df_clean["steam_language_count"] == 103,
        [
            "steam_name",
            "steam_language_count",
            "feature_language_count_parsed"
        ]
    ].head(10)
)


Successfully parsed games: 87852
Missing or unparseable entries: 7845
Count mismatches: 1

Games originally recorded with 103 languages:


,steam_name,steam_language_count,feature_language_count_parsed
1250,Zup! S,103,NaN
1583,Zup! F,103,NaN
1829,Zup! 2,103,NaN
1890,Zup! X,103,NaN
2064,Zup! 3,103,NaN
2578,Zup! 8,103,NaN
2613,Zup! Z,103,NaN
3062,UBERMOSH,103,NaN
3151,Zup! 4,103,NaN
3296,Zup! 5,103,NaN


In [51]:
# Investigate records that failed language parsing
examples = df_clean.loc[
    df_clean["steam_language_count"] == 103,
    ["steam_name", "steam_supported_languages"]
].head(3)

for _, row in examples.iterrows():
    raw = str(row["steam_supported_languages"])

    print("\nGame:", row["steam_name"])
    print("String length:", len(raw))
    print("First 120 characters:", repr(raw[:120]))
    print("Last 120 characters:", repr(raw[-120:]))
    print("Ends with ]:", raw.rstrip().endswith("]"))

    try:
        parsed = ast.literal_eval(raw)
        print("Parsing successful:", len(parsed))
    except (ValueError, SyntaxError) as e:
        print("Parsing error:", str(e)[:200])

# Identify the single count mismatch
mismatch_mask = (
    df_clean["feature_language_count_parsed"].notna()
    & (
        df_clean["steam_language_count"]
        != df_clean["feature_language_count_parsed"]
    )
)

print("\nRecords with count mismatches:")

display(
    df_clean.loc[
        mismatch_mask,
        [
            "steam_name",
            "steam_language_count",
            "feature_language_count_parsed",
            "steam_supported_languages"
        ]
    ]
)



Game: Zup! S
String length: 1216
First 120 characters: "['English', 'Russian', 'French', 'Italian', 'German', 'Spanish - Spain', 'Arabic', 'Bulgarian', 'Portuguese - Brazil', '"
Last 120 characters: "swana', 'Turkmen', 'Uzbek', 'Uyghur', 'Urdu', 'Filipino', 'Hausa', 'Hindi', 'Croatian', 'Cherokee', 'Scots', 'Estonian']"
Ends with ]: True
Parsing error: invalid syntax (<unknown>, line 1)

Game: Zup! F
String length: 1216
First 120 characters: "['English', 'Russian', 'French', 'Italian', 'German', 'Spanish - Spain', 'Arabic', 'Bulgarian', 'Portuguese - Brazil', '"
Last 120 characters: "swana', 'Turkmen', 'Uzbek', 'Uyghur', 'Urdu', 'Filipino', 'Hausa', 'Hindi', 'Croatian', 'Cherokee', 'Scots', 'Estonian']"
Ends with ]: True
Parsing error: invalid syntax (<unknown>, line 1)

Game: Zup! 2
String length: 1216
First 120 characters: "['English', 'French', 'Italian', 'German', 'Spanish - Spain', 'Arabic', 'Bulgarian', 'Portuguese - Brazil', 'Hungarian',"
Last 120 characters: "swana', 'Turkmen'

,steam_name,steam_language_count,feature_language_count_parsed,steam_supported_languages
11498,Codename: Panzers - Cold War,9,8.0,"['English', 'French', 'German', 'Spanish - Spa..."


In [52]:

def parse_supported_languages(value):
    if pd.isna(value) or not str(value).strip():
        return np.nan, "missing"

    text = str(value).strip()

    # First attempt: standard Python list parsing
    try:
        languages = ast.literal_eval(text)

        if isinstance(languages, (list, tuple)):
            unique = {
                str(x).strip()
                for x in languages
                if str(x).strip()
            }
            return len(unique), "standard"

    except (ValueError, SyntaxError):
        pass

    # Fallback: parse quoted list entries
    if not (text.startswith("[") and text.endswith("]")):
        return np.nan, "unparseable"

    inner = text[1:-1].strip()

    if not inner:
        return 0, "empty"

    quote = inner[0]

    if quote not in ("'", '"') or not inner.endswith(quote):
        return np.nan, "unparseable"

    parts = re.split(
        re.escape(quote) + r"\s*,\s*" + re.escape(quote),
        inner[1:-1]
    )

    parts = [p.strip() for p in parts]

    if not all(parts):
        return np.nan, "unparseable"

    return len(set(parts)), "fallback"


# Apply corrected parsing
language_results = (
    df_clean["steam_supported_languages"]
    .apply(parse_supported_languages)
)

df_clean[
    ["feature_language_count", "language_parse_status"]
] = pd.DataFrame(
    language_results.tolist(),
    index=df_clean.index
)

# Preserve an indicator for unavailable information
df_clean["flag_missing_language_metadata"] = (
    df_clean["feature_language_count"].isna()
)

print("Language feature reconstruction completed.")


Language feature reconstruction completed.


In [53]:

# Summarize parsing results
print("Parsing status:")
display(df_clean["language_parse_status"].value_counts())

print("\nCorrected language count summary:")
display(df_clean["feature_language_count"].describe())

# Check the previously problematic games
print("\nGames originally recorded with 103 languages:")

display(
    df_clean.loc[
        df_clean["steam_language_count"] == 103,
        [
            "steam_name",
            "steam_language_count",
            "feature_language_count",
            "language_parse_status"
        ]
    ].head(10)
)

# Identify remaining count disagreements
valid = df_clean["feature_language_count"].notna()

differences = df_clean.loc[
    valid & (
        df_clean["steam_language_count"]
        != df_clean["feature_language_count"]
    ),
    [
        "steam_name",
        "steam_language_count",
        "feature_language_count",
        "steam_supported_languages"
    ]
]

print("\nRemaining count disagreements:", len(differences))
display(differences.head(10))

assert df_clean["feature_language_count"].dropna().ge(0).all()

print("\nLanguage feature validation completed.")


Parsing status:


language_parse_status
standard    94141
fallback     1556
Name: count, dtype: int64


Corrected language count summary:


count    95697.000000
mean         5.317324
std         13.563018
min          0.000000
25%          1.000000
50%          1.000000
75%          5.000000
max        102.000000
Name: feature_language_count, dtype: float64


Games originally recorded with 103 languages:


,steam_name,steam_language_count,feature_language_count,language_parse_status
1250,Zup! S,103,102,fallback
1583,Zup! F,103,102,fallback
1829,Zup! 2,103,102,fallback
1890,Zup! X,103,102,fallback
2064,Zup! 3,103,102,fallback
2578,Zup! 8,103,102,fallback
2613,Zup! Z,103,102,fallback
3062,UBERMOSH,103,102,fallback
3151,Zup! 4,103,102,fallback
3296,Zup! 5,103,102,fallback



Remaining count disagreements: 1557


,steam_name,steam_language_count,feature_language_count,steam_supported_languages
1250,Zup! S,103,102,"['English', 'Russian', 'French', 'Italian', 'G..."
1583,Zup! F,103,102,"['English', 'Russian', 'French', 'Italian', 'G..."
1829,Zup! 2,103,102,"['English', 'French', 'Italian', 'German', 'Sp..."
1890,Zup! X,103,102,"['English', 'French', 'Italian', 'German', 'Sp..."
2064,Zup! 3,103,102,"['English', 'French', 'Italian', 'German', 'Sp..."
2189,Barro,102,101,"['English', 'French', 'Italian', 'German', 'Sp..."
2578,Zup! 8,103,102,"['English', 'French', 'Italian', 'German', 'Sp..."
2613,Zup! Z,103,102,"['English', 'French', 'Italian', 'German', 'Sp..."
3062,UBERMOSH,103,102,"['English', 'French', 'Italian', 'German', 'Sp..."
3151,Zup! 4,103,102,"['English', 'French', 'Italian', 'German', 'Sp..."



Language feature validation completed.


In [54]:

# Step 4: Finalize language feature engineering

# Examine differences between original and reconstructed counts
df_clean["language_count_difference"] = (
    df_clean["steam_language_count"]
    - df_clean["feature_language_count"]
)

print("Original vs corrected count differences:")
display(
    pd.crosstab(
        df_clean["language_parse_status"],
        df_clean["language_count_difference"]
    )
)

# Zero recorded languages may indicate unavailable metadata
df_clean["flag_zero_languages"] = (
    df_clean["feature_language_count"] == 0
)

# Preserve missingness rather than treating zero as confirmed support
df_clean["feature_language_count_model"] = (
    df_clean["feature_language_count"]
    .mask(df_clean["flag_zero_languages"])
)

# Log transform the usable counts
df_clean["feature_log_language_count"] = np.log1p(
    df_clean["feature_language_count_model"]
)

print("\nMissing language information:",
      df_clean["flag_zero_languages"].sum())

print("\nLog language count summary:")
display(
    df_clean["feature_log_language_count"].describe()
)

# Validation
assert df_clean["feature_language_count"].notna().all()

assert (
    df_clean["feature_log_language_count"]
    .dropna()
    .ge(0)
    .all()
)

print("\nLanguage feature engineering finalized.")


Original vs corrected count differences:


language_count_difference,0,1
language_parse_status,,
fallback,0,1556
standard,94140,1



Missing language information: 6289

Log language count summary:


count    89408.000000
mean         1.311200
std          0.863723
min          0.693147
25%          0.693147
50%          0.693147
75%          1.791759
max          4.634729
Name: feature_log_language_count, dtype: float64


Language feature engineering finalized.


### 1.5 Modelling Inclusion Thresholds

Inclusion thresholds were established before statistical modelling to ensure consistent sample selection and improve data reliability.

For the player reception model, games were required to have at least one Steam review, resulting in 63,554 eligible observations. Free-to-play games were retained, and no HLTB requirements were applied.

For the completion model, games were required to be classified as story-driven, have a valid positive completion ratio, at least 50 Steam reviews, and at least 5 HLTB main-story polls.

Applying these criteria reduced the completion sample from 14,863 story-driven games to **2,357 eligible observations (2.46% of the cleaned dataset)**.

Eligibility flags were created for both models without removing additional records from the cleaned dataset. All validation checks passed.

These thresholds were fixed before model fitting to maintain consistent sample selection. However, the restrictive completion-model criteria may introduce selection bias, which should be considered during subsequent analysis.

In [55]:

# Step 5: Define model inclusion thresholds

MIN_REVIEWS_RECEPTION = 10
MIN_REVIEWS_COMPLETION = 50
MIN_HLTB_POLLS = 5

# Convert HLTB poll counts to numeric
hltb_polls = pd.to_numeric(
    df_clean["hltb_main_story_polled"],
    errors="coerce"
)

# Reception model eligibility
df_clean["flag_reception_model"] = (
    df_clean["steam_review_count"]
    >= MIN_REVIEWS_RECEPTION
)

# Completion model eligibility
df_clean["flag_completion_model"] = (
    df_clean["flag_completion_eligible"]
    & (
        df_clean["steam_review_count"]
        >= MIN_REVIEWS_COMPLETION
    )
    & (hltb_polls >= MIN_HLTB_POLLS)
)

# Construct a cumulative selection funnel
completion_base = df_clean["flag_completion_eligible"]

funnel = pd.DataFrame({
    "Stage": [
        "Cleaned dataset",
        "Story-driven games",
        "Valid completion ratio",
        "At least 50 Steam reviews",
        "At least 5 HLTB polls"
    ],
    "Games": [
        len(df_clean),
        int(df_clean["flag_story_driven"].sum()),
        int(completion_base.sum()),
        int((
            completion_base
            & (df_clean["steam_review_count"] >= 50)
        ).sum()),
        int(df_clean["flag_completion_model"].sum())
    ]
})

funnel["Percentage of cleaned (%)"] = (
    funnel["Games"] / len(df_clean) * 100
).round(2)

display(funnel)

print(
    "Reception model sample:",
    f"{df_clean['flag_reception_model'].sum():,}"
)

print(
    "Completion model sample:",
    f"{df_clean['flag_completion_model'].sum():,}"
)

# Validation checks

assert (
    df_clean["flag_reception_model"]
    == df_clean["steam_review_count"].ge(MIN_REVIEWS_RECEPTION)
).all()

assert df_clean["flag_completion_model"].sum() <= 3508
assert funnel["Games"].is_monotonic_decreasing

print("\nInclusion threshold checks passed.")


,Stage,Games,Percentage of cleaned (%)
0,Cleaned dataset,95697,100.00
1,Story-driven games,14863,15.53
2,Valid completion ratio,3508,3.67
3,At least 50 Steam reviews,3447,3.60
4,At least 5 HLTB polls,2357,2.46


Reception model sample: 43,669
Completion model sample: 2,357

Inclusion threshold checks passed.


In [56]:

# Step 5: Selection bias check — HLTB matched vs unmatched

selection_variables = [
    "steam_price",
    "steam_review_count",
    "steam_positive_rate",
    "steam_years_since_release"
]

hltb_selection_summary = (
    df_clean.assign(
        HLTB_Group=np.where(
            df_clean["matched_hltb"].fillna(False),
            "HLTB Matched",
            "HLTB Unmatched"
        )
    )
    .groupby("HLTB_Group")
    .agg(
        Games=("steam_appid", "size"),
        Median_Price=("steam_price", "median"),
        Median_Review_Count=("steam_review_count", "median"),
        Median_Positive_Rate=("steam_positive_rate", "median"),
        Median_Game_Age=("steam_years_since_release", "median")
    )
    .round(3)
    .reset_index()
)

display(hltb_selection_summary)

# Validate that every game is included
assert hltb_selection_summary["Games"].sum() == len(df_clean)

print("HLTB selection bias comparison checks passed.")


,HLTB_Group,Games,Median_Price,Median_Review_Count,Median_Positive_Rate,Median_Game_Age
0,HLTB Matched,16653,3.49,197.0,0.846,6.0
1,HLTB Unmatched,79044,1.99,4.0,0.805,3.0


HLTB selection bias comparison checks passed.


In [57]:

# Step 5: Selection bias check — Completion model eligible vs excluded

completion_selection_summary = (
    df_clean.assign(
        Completion_Group=np.where(
            df_clean["flag_completion_model"],
            "Completion Eligible",
            "Completion Excluded"
        )
    )
    .groupby("Completion_Group")
    .agg(
        Games=("steam_appid", "size"),
        Median_Price=("steam_price", "median"),
        Median_Review_Count=("steam_review_count", "median"),
        Median_Positive_Rate=("steam_positive_rate", "median"),
        Median_Game_Age=("steam_years_since_release", "median")
    )
    .round(3)
    .reset_index()
)

display(completion_selection_summary)

# Validate group sizes
assert completion_selection_summary["Games"].sum() == len(df_clean)

assert (
    completion_selection_summary.loc[
        completion_selection_summary["Completion_Group"] == "Completion Eligible",
        "Games"
    ].iloc[0]
    == df_clean["flag_completion_model"].sum()
)

print("Completion selection bias comparison checks passed.")


,Completion_Group,Games,Median_Price,Median_Review_Count,Median_Positive_Rate,Median_Game_Age
0,Completion Eligible,2357,4.99,1477.0,0.874,7.0
1,Completion Excluded,93340,2.35,6.0,0.818,3.0


Completion selection bias comparison checks passed.


### 1.6 Training and Validation Split

The cleaned dataset of 95,697 games was divided into development-training (80%) and validation (20%) subsets using stratified random sampling. Stratification preserved approximately equal proportions of reception-eligible and completion-eligible games across both datasets, while a fixed random seed ensured reproducibility.

The resulting development-training dataset contained 76,557 games, while the validation dataset contained 19,140 games. Validation checks confirmed that all observations were accounted for and that no games appeared in both subsets.

Missing language counts were subsequently handled using median imputation. The median of 1 language was calculated exclusively from the development-training dataset and applied consistently to both subsets to prevent data leakage.

This process replaced 5,053 missing values in the development-training dataset and 1,236 in the validation dataset. An additional indicator was retained to identify originally missing observations.

All imputation checks passed, confirming that no missing values remained in the imputed language-count features. The original held-out test dataset from Notebook 00 remained untouched.

In [58]:

# Step 6: Split training data into development and validation sets

from sklearn.model_selection import train_test_split

VALIDATION_SIZE = 0.20

# Validate eligibility relationship
assert not (
    df_clean["flag_completion_model"]
    & ~df_clean["flag_reception_model"]
).any()

# Stratify into three groups:
# 0 = Neither model
# 1 = Reception only
# 2 = Reception and completion
strata = (
    df_clean["flag_reception_model"].astype(int)
    + df_clean["flag_completion_model"].astype(int)
)

# Create development and validation datasets
df_train, df_val = train_test_split(
    df_clean,
    test_size=VALIDATION_SIZE,
    random_state=config.SEED,
    stratify=strata
)

df_train = df_train.copy()
df_val = df_val.copy()

# Summarize sample sizes
split_summary = pd.DataFrame({
    "Dataset": ["Development Train", "Validation"],
    "Total Games": [
        len(df_train),
        len(df_val)
    ],
    "Reception Eligible": [
        int(df_train["flag_reception_model"].sum()),
        int(df_val["flag_reception_model"].sum())
    ],
    "Completion Eligible": [
        int(df_train["flag_completion_model"].sum()),
        int(df_val["flag_completion_model"].sum())
    ]
})

display(split_summary)

# Validation checks
assert len(df_train) + len(df_val) == len(df_clean)

assert set(df_train["steam_appid"]).isdisjoint(
    set(df_val["steam_appid"])
)

assert (
    df_train["flag_reception_model"].sum()
    + df_val["flag_reception_model"].sum()
    == df_clean["flag_reception_model"].sum()
)

assert (
    df_train["flag_completion_model"].sum()
    + df_val["flag_completion_model"].sum()
    == df_clean["flag_completion_model"].sum()
)

print("\nTraining-validation split checks passed.")


,Dataset,Total Games,Reception Eligible,Completion Eligible
0,Development Train,76557,34935,1886
1,Validation,19140,8734,471



Training-validation split checks passed.


In [59]:

# Developer game count (training split only)
from is630.features import extract_primary_developer

# Identify primary developers in both datasets
for dataset in [df_train, df_val]:
    dataset["developer_group"] = extract_primary_developer(
        dataset["steam_developers"],
        dataset["steam_appid"]
    )

# Calculate developer game counts using training data only
developer_game_counts = (
    df_train["developer_group"].value_counts()
)

# Apply training-derived counts to both datasets
for dataset in [df_train, df_val]:
    dataset["feature_log_dev_game_count"] = np.log1p(
        dataset["developer_group"]
        .map(developer_game_counts)
        .fillna(0)
    )

# Validation
assert df_train["feature_log_dev_game_count"].notna().all()
assert df_val["feature_log_dev_game_count"].notna().all()

assert df_train["feature_log_dev_game_count"].ge(0).all()
assert df_val["feature_log_dev_game_count"].ge(0).all()

print("Developer game count features created successfully.")


<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal


Developer game count features created successfully.


In [60]:

# Check website and support contact metadata

support_columns = [
    "steam_website",
    "steam_support_url",
    "steam_support_email"
]

print("Data types:")
print(df_clean[support_columns].dtypes)

print("\nExample values:")
display(df_clean[support_columns].head(10))

print("\nMissing values:")
print(df_clean[support_columns].isna().sum())


Data types:
steam_website          object
steam_support_url      object
steam_support_email    object
dtype: object

Example values:


,steam_website,steam_support_url,steam_support_email
0,http://counter-strike.net/,None,None
1,http://www.dota2.com/,None,None
2,https://www.rockstargames.com/V/,https://support.rockstargames.com,None
3,http://www.terraria.org/,https://forums.terraria.org/index.php?forums/p...,support@terraria.org
4,http://rainbow6.ubi.com/,http://support.ubi.com,None
5,http://rust.facepunch.com/,http://support.facepunchstudios.com/,None
6,http://www.teamfortress.com/,http://steamcommunity.com/app/440,None
7,https://gmod.facepunch.com,https://support.facepunchstudios.com/,None
8,https://www.gamesci.cn/wukong,None,cs@gamesci.com.cn
9,https://www.playstation.com/games/helldivers-2...,None,support@arrowhead.zendesk.com



Missing values:
steam_website          56835
steam_support_url      53607
steam_support_email    16990
dtype: int64


In [61]:

# Step 6: Handle missing language values
# Use training-set statistics to prevent data leakage

# Calculate median from development-training data only
train_language_median = (
    df_train["feature_language_count_model"].median()
)

print(
    f"Training language-count median: "
    f"{train_language_median:.2f}"
)

# Apply the same training median to both datasets
for dataset in [df_train, df_val]:

    # Flag observations with missing language information
    dataset["feature_language_missing"] = (
        dataset["feature_language_count_model"]
        .isna()
        .astype(int)
    )

    # Impute missing values
    dataset["feature_language_count_imputed"] = (
        dataset["feature_language_count_model"]
        .fillna(train_language_median)
    )

    # Apply logarithmic transformation
    dataset["feature_log_language_count_imputed"] = (
        np.log1p(
            dataset["feature_language_count_imputed"]
        )
    )

# Summarize imputation results
imputation_summary = pd.DataFrame({
    "Dataset": ["Development Train", "Validation"],
    "Missing Before": [
        df_train["feature_language_count_model"].isna().sum(),
        df_val["feature_language_count_model"].isna().sum()
    ],
    "Missing After": [
        df_train["feature_language_count_imputed"].isna().sum(),
        df_val["feature_language_count_imputed"].isna().sum()
    ],
    "Imputed Median": [
        train_language_median,
        train_language_median
    ]
})

display(imputation_summary)

# Validation checks
assert df_train[
    "feature_language_count_imputed"
].notna().all()

assert df_val[
    "feature_language_count_imputed"
].notna().all()

assert df_train[
    "feature_log_language_count_imputed"
].notna().all()

assert df_val[
    "feature_log_language_count_imputed"
].notna().all()

print("\nLanguage imputation checks passed.")


Training language-count median: 1.00


,Dataset,Missing Before,Missing After,Imputed Median
0,Development Train,5009,0,1.0
1,Validation,1280,0,1.0



Language imputation checks passed.


In [62]:

# Step 6: Final validation of model-ready features

# Define common predictor variables

MODEL_FEATURES = [
    # Pricing
    "feature_log_price",
    "flag_free_to_play",

    # Game age and release window
    "steam_years_since_release",
    "flag_q4_release",

    # Platform support
    "steam_mac",
    "steam_linux",

    # Language and audio localization
    "feature_log_language_count_imputed",
    "feature_language_missing",
    "feature_log_audio_language_count",
    "flag_has_full_audio",

    # In-game and store features
    "feature_log_achievements",
    "flag_has_achievements",
    "flag_age_restricted",
    "cat_singleplayer",
    "cat_multiplayer",
    "cat_coop",
    "cat_controller",
    "cat_steam_cloud",
    "cat_trading_cards",
    "cat_vr",

    # Genres
    "genre_indie",
    "genre_casual",
    "genre_action",
    "genre_adventure",
    "genre_simulation",
    "genre_strategy",
    "genre_rpg",
    "genre_early_access",
    "flag_missing_genre",

    # Narrative
    "flag_story_driven",
    "flag_missing_tags",

    # Studio and publishing
    "flag_self_published",
    "feature_log_dev_game_count",

    # Storefront and support
    "flag_has_website",
    "flag_has_support_contact"
]


# HLTB scope features for matched-subset analyses
HLTB_SCOPE_FEATURES = [
    "feature_log_main_story_hours",
    "flag_has_side_content"
]

validation_results = []

for name, dataset in [
    ("Development Train", df_train),
    ("Validation", df_val)
]:
    # Check all required predictors exist
    missing_cols = set(MODEL_FEATURES) - set(dataset.columns)
    assert not missing_cols, f"Missing columns: {missing_cols}"

    # Check numerical values
    X = dataset[MODEL_FEATURES].apply(
        pd.to_numeric, errors="coerce"
    )

    missing_values = int(X.isna().sum().sum())
    infinite_values = int(
        np.isinf(X.to_numpy(dtype=float)).sum()
    )

    # Check targets in their eligible samples
    reception = dataset.loc[
        dataset["flag_reception_model"],
        "steam_positive_rate"
    ]

    completion = dataset.loc[
        dataset["flag_completion_model"],
        "completion_ratio"
    ]

    assert reception.notna().all()
    assert reception.between(0, 1).all()

    assert completion.notna().all()
    assert completion.gt(0).all()

    assert missing_values == 0
    assert infinite_values == 0

    validation_results.append({
        "Dataset": name,
        "Games": len(dataset),
        "Predictors": len(MODEL_FEATURES),
        "Missing Predictor Values": missing_values,
        "Infinite Predictor Values": infinite_values,
        "Reception Eligible": len(reception),
        "Completion Eligible": len(completion)
    })

validation_summary = pd.DataFrame(validation_results)

display(validation_summary)

# Ensure no targets are included as predictors
TARGET_COLUMNS = [
    "steam_review_count",
    "steam_positive_rate",
    "completion_ratio"
]

assert not set(MODEL_FEATURES).intersection(TARGET_COLUMNS)

# Check that game IDs do not overlap
assert set(df_train["steam_appid"]).isdisjoint(
    set(df_val["steam_appid"])
)

print("\nAll model-ready feature checks passed.")


,Dataset,Games,Predictors,Missing Predictor Values,Infinite Predictor Values,Reception Eligible,Completion Eligible
0,Development Train,76557,35,0,0,34935,1886
1,Validation,19140,35,0,0,8734,471



All model-ready feature checks passed.


### 1.7 Model-Ready Data and Feature Specification

Final validation checks were conducted to ensure that the development-training and validation datasets were suitable for statistical modelling.

Both datasets contained 12 engineered predictors, with no missing or infinite values. The development-training dataset contained 76,557 games, including 50,843 reception-eligible and 1,886 completion-eligible observations. The validation dataset contained 19,140 games, including 12,711 reception-eligible and 471 completion-eligible observations.

The processed datasets will be saved as Parquet files to preserve data types and support efficient loading in Notebook 03. A separate feature specification will document the selected predictors, target variables, inclusion thresholds, and preprocessing parameters.

These outputs will ensure consistent modelling and reproducibility while preserving the original held-out test dataset for final evaluation.

In [63]:

# Step 7: Save model-ready datasets

# Create output directory if necessary
config.PROCESSED.mkdir(parents=True, exist_ok=True)

# Define output paths
train_path = config.PROCESSED / "02_train.parquet"
val_path = config.PROCESSED / "02_validation.parquet"

# Remove duplicate column names before exporting
df_train = df_train.loc[:, ~df_train.columns.duplicated()].copy()
df_val = df_val.loc[:, ~df_val.columns.duplicated()].copy()

# Verify column names are unique
assert df_train.columns.is_unique
assert df_val.columns.is_unique

print("Duplicate column checks passed.")


# Save development-training and validation datasets
df_train.to_parquet(train_path, index=False)
df_val.to_parquet(val_path, index=False)

# Reload saved files to verify successful export
train_check = pd.read_parquet(train_path)
val_check = pd.read_parquet(val_path)

# Verify row counts and column consistency
assert len(train_check) == len(df_train)
assert len(val_check) == len(df_val)

assert set(train_check.columns) == set(val_check.columns)
assert set(MODEL_FEATURES).issubset(train_check.columns)

# Confirm no games overlap between datasets
assert set(train_check["steam_appid"]).isdisjoint(
    set(val_check["steam_appid"])
)

print("Model-ready datasets saved successfully.\n")

print("Development training:")
print(f"Path: {train_path}")
print(f"Shape: {train_check.shape}")

print("\nValidation:")
print(f"Path: {val_path}")
print(f"Shape: {val_check.shape}")

print("\nDataset export checks passed.")


Duplicate column checks passed.
Model-ready datasets saved successfully.

Development training:
Path: C:\Users\sheen\Documents\SMU\Courses\IS630 Statistical Thinking with Data Science\Group Assignment\IS630\data\processed\02_train.parquet
Shape: (76557, 145)

Validation:
Path: C:\Users\sheen\Documents\SMU\Courses\IS630 Statistical Thinking with Data Science\Group Assignment\IS630\data\processed\02_validation.parquet
Shape: (19140, 145)

Dataset export checks passed.


In [64]:

# Step 7: Save feature specification

import json

feature_spec = {
    "model_features": MODEL_FEATURES,
    "hltb_scope_features": HLTB_SCOPE_FEATURES,

    "targets": {
        "reception": "steam_positive_rate",
        "review_count": "steam_review_count",
        "completion": "completion_ratio"
    },

    "eligibility_flags": {
        "reception": "flag_reception_model",
        "completion": "flag_completion_model"
    },

    "inclusion_thresholds": {
        "min_reviews_reception": MIN_REVIEWS_RECEPTION,
        "min_reviews_completion": MIN_REVIEWS_COMPLETION,
        "min_hltb_polls": MIN_HLTB_POLLS,
        "story_driven_required_for_completion": True
    },

    "preprocessing": {
        "language_imputation": "median",
        "language_imputation_value": float(
            train_language_median
        ),
        "language_missing_indicator": "feature_language_missing",
        "language_predictor": "feature_log_language_count_imputed",
        "price_transformation": "log1p",
        "genre_encoding": "multi_label_binary",
        "genre_mapping": GENRE_MAP,
        "price_band_labels": PRICE_LABELS,
        "price_band_edges": [
            float(x) if np.isfinite(x) else "inf"
            for x in PRICE_BINS
        ],
        "game_age_reference_year": config.SNAPSHOT_YEAR
    },

    "validation_split": {
        "validation_fraction": VALIDATION_SIZE,
        "random_seed": config.SEED,
        "stratification": "reception_and_completion_eligibility"
    },

    "dataset_files": {
        "train": train_path.name,
        "validation": val_path.name
    }
}

# Save specification as JSON
spec_path = config.PROCESSED / "02_feature_spec.json"

with open(spec_path, "w", encoding="utf-8") as f:
    json.dump(feature_spec, f, indent=4, allow_nan=False)

# Reload to verify successful export
with open(spec_path, "r", encoding="utf-8") as f:
    saved_spec = json.load(f)

assert saved_spec["model_features"] == MODEL_FEATURES

print("Feature specification saved successfully.")
print(f"Path: {spec_path}")

print("\nNumber of predictors:",
      len(saved_spec["model_features"]))

print("Language imputation value:",
      saved_spec["preprocessing"]["language_imputation_value"])

print("\nFeature specification checks passed.")


Feature specification saved successfully.
Path: C:\Users\sheen\Documents\SMU\Courses\IS630 Statistical Thinking with Data Science\Group Assignment\IS630\data\processed\02_feature_spec.json

Number of predictors: 35
Language imputation value: 1.0

Feature specification checks passed.
